# Bronze Layer

## Data Optimization

Purpose:

- Optimize Bronze datasets
- Improve read performance
- Reduce the number of small files
- Prepare datasets for downstream processing

## Environment Bootstrap

Required on Databricks Free Edition: there is no `libraries: - whl: ...` mechanism wired to a classic/serverless cluster here, so the project wheel has to be pip-installed explicitly in the notebook before any `data_platform`/`integrations` import works.

`wheel_path` comes from a Job base_parameter (`${workspace.root_path}/artifacts/.internal`, resolved by the Databricks bundle at deploy time) -- portable across users and targets (dev/prod), unlike a hardcoded `/Workspace/Users/<you>/...` path. Bundle substitutions only expand inside bundle YAML files, never inside notebook content directly, which is why this goes through a widget instead of being inlined here.

In [ ]:
dbutils.widgets.text("wheel_path", "")
wheel_path = dbutils.widgets.get("wheel_path")
wheel_glob = f"{wheel_path}/*.whl"

%pip install $wheel_glob

dbutils.library.restartPython()

## Imports

In [ ]:
from data_platform.compute.delta_io import read_delta, write_delta
from data_platform.compute.spark import get_spark
from data_platform.storage.config import StorageConfig
from integrations.databricks.runtime.parameters import get_parameter

## Parameters

In [ ]:
entity = get_parameter("entity", default="customers")

## Spark Session

In [ ]:
spark = get_spark("Bronze Optimization")

## Read Bronze Dataset

In [ ]:
bronze_df = read_delta(spark, StorageConfig.bronze(entity))

## Display Dataset Information

In [ ]:
bronze_df.printSchema()

bronze_df.show(10)

file_count = spark.sql(
    f"DESCRIBE DETAIL delta.`{StorageConfig.bronze(entity)}`"
).first()["numFiles"]

print(f"Files: {file_count}")

## Optimize Dataset Partitions

In [ ]:
optimized_df = bronze_df.coalesce(1)

## Write Optimized Dataset

In [ ]:
write_delta(optimized_df, StorageConfig.bronze(entity), mode="overwrite")

## Validate Optimized Dataset

In [ ]:
optimized_validation_df = read_delta(spark, StorageConfig.bronze(entity))

print(f"Total records: {optimized_validation_df.count()}")

optimized_file_count = spark.sql(
    f"DESCRIBE DETAIL delta.`{StorageConfig.bronze(entity)}`"
).first()["numFiles"]

print(f"Files: {optimized_file_count}")

## Optimization Summary

In [ ]:
print("=" * 80)

print("Bronze Optimization Completed")

print("=" * 80)